In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

df = pd.read_csv("data.csv")

print("Dataset Shape:", df.shape)
display(df.head())
display(df.info())

Dataset Shape: (4999, 20)


,rank,bgg_url,game_id,names,min_players,max_players,avg_time,min_time,max_time,year,avg_rating,geek_rating,num_votes,image_url,age,mechanic,owned,category,designer,weight
0,1,https://boardgamegeek.com/boardgame/174430/glo...,174430,Gloomhaven,1,4,120,60,120,2017,8.98893,8.61858,15376,https://cf.geekdo-images.com/original/img/lDN3...,12,"Action / Movement Programming, Co-operative Pl...",25928,"Adventure, Exploration, Fantasy, Fighting, Min...",Isaac Childres,3.7543
1,2,https://boardgamegeek.com/boardgame/161936/pan...,161936,Pandemic Legacy: Season 1,2,4,60,60,60,2015,8.66140,8.50163,26063,https://cf.geekdo-images.com/original/img/P_Sw...,13,"Action Point Allowance System, Co-operative Pl...",41605,"Environmental, Medical","Rob Daviau, Matt Leacock",2.8210
2,3,https://boardgamegeek.com/boardgame/182028/thr...,182028,Through the Ages: A New Story of Civilization,2,4,240,180,240,2015,8.60673,8.30183,12352,https://cf.geekdo-images.com/original/img/1d2h...,14,"Action Point Allowance System, Auction/Bidding...",15848,"Card Game, Civilization, Economic",Vlaada Chvátil,4.3678
3,4,https://boardgamegeek.com/boardgame/167791/ter...,167791,Terraforming Mars,1,5,120,120,120,2016,8.38461,8.19914,26004,https://cf.geekdo-images.com/original/img/o8z_...,12,"Card Drafting, Hand Management, Set Collection...",33340,"Economic, Environmental, Industry / Manufactur...",Jacob Fryxelius,3.2456
4,5,https://boardgamegeek.com/boardgame/12333/twil...,12333,Twilight Struggle,2,2,180,120,180,2005,8.33954,8.19787,31301,https://cf.geekdo-images.com/original/img/ZPnn...,13,"Area Control / Area Influence, Campaign / Batt...",42952,"Modern Warfare, Political, Wargame","Ananda Gupta, Jason Matthews",3.5518


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 20 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   rank         4999 non-null   int64  
 1   bgg_url      4999 non-null   object 
 2   game_id      4999 non-null   int64  
 3   names        4999 non-null   object 
 4   min_players  4999 non-null   int64  
 5   max_players  4999 non-null   int64  
 6   avg_time     4999 non-null   int64  
 7   min_time     4999 non-null   int64  
 8   max_time     4999 non-null   int64  
 9   year         4999 non-null   int64  
 10  avg_rating   4999 non-null   float64
 11  geek_rating  4999 non-null   float64
 12  num_votes    4999 non-null   int64  
 13  image_url    4998 non-null   object 
 14  age          4999 non-null   int64  
 15  mechanic     4999 non-null   object 
 16  owned        4999 non-null   int64  
 17  category     4999 non-null   object 
 18  designer     4999 non-null   object 
 19  weight

None

Data cleaning + feature engineering


In [15]:
#This step removes irrelevant columns and handles basic data issues
df = df.drop(columns=["bgg_url", "image_url"])

df = df.drop_duplicates()

df = df.dropna(subset=[
    "avg_rating",
    "year",
    "mechanic",
    "category",
    "designer",
    "weight"
])

df = df[df["num_votes"] >= 50]

print("Dataset Shape After Cleaning:", df.shape)

Dataset Shape After Cleaning: (4999, 18)


In [16]:
#This section ensures that numeric columns are properly converted and that invalid values are coerced into NaN and removed if necessary.
numeric_cols = [
    "min_players",
    "max_players",
    "avg_time",
    "min_time",
    "max_time",
    "year",
    "avg_rating",
    "geek_rating",
    "num_votes",
    "owned",
    "age",
    "weight"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna()

In [17]:
#This section ensures the dataset satisfies that constraint and optionally removes extreme outliers that may distort training.
df = df[df["avg_time"] > 0]
df = df[df["weight"].between(0, 5)]

print("Final dataset size:", len(df))

Final dataset size: 4984


In [18]:
#This section creates additional useful features derived from the original metadata
current_year = pd.Timestamp.now().year

df["player_range"] = df["max_players"] - df["min_players"]
df["game_age"] = current_year - df["year"]
df["rating_density"] = df["num_votes"] / df["owned"]

df["rating_density"] = df["rating_density"].replace([np.inf, -np.inf], np.nan)
df["rating_density"] = df["rating_density"].fillna(0)

In [19]:
#Mechanics are stored as comma-separated strings representing gameplay systems.
#This section converts them into lists and then applies multi-label binarization so that each mechanic becomes a binary feature usable by machine learning models.
df["mechanic_list"] = df["mechanic"].apply(
    lambda x: [m.strip() for m in x.split(",")]
)

mlb_mech = MultiLabelBinarizer()

mechanic_features = pd.DataFrame(
    mlb_mech.fit_transform(df["mechanic_list"]),
    columns=mlb_mech.classes_,
    index=df.index
)

df = pd.concat([df, mechanic_features], axis=1)

In [20]:
#Categories represent broader themes of the game such as fantasy, strategy, or war. Similar to mechanics, they are encoded into binary variables so they can be used as features for prediction models.
df["category_list"] = df["category"].apply(
    lambda x: [c.strip() for c in x.split(",")]
)

mlb_cat = MultiLabelBinarizer()

category_features = pd.DataFrame(
    mlb_cat.fit_transform(df["category_list"]),
    columns=mlb_cat.classes_,
    index=df.index
)

df = pd.concat([df, category_features], axis=1)

In [21]:
#This section calculates statistics describing each designer's historical performance, including average rating, number of games produced, and total votes across all their games.
designer_stats = df.groupby("designer").agg(
    designer_avg_rating=("avg_rating", "mean"),
    designer_num_games=("game_id", "count"),
    designer_total_votes=("num_votes", "sum")
)

df = df.merge(designer_stats, on="designer", how="left")

In [22]:
#This step measures how common each mechanic is across the dataset
mechanic_counts = df["mechanic_list"].explode().value_counts()

def mechanic_popularity(mechanics):
    return np.mean([mechanic_counts[m] for m in mechanics])

df["mechanic_popularity"] = df["mechanic_list"].apply(mechanic_popularity)

In [23]:
#This section prepares the dataset for machine learning by defining the target variable and removing columns that should not be used as model inputs (such as text fields or identifiers).
target = "avg_rating"

drop_cols = [
    "rank",
    "game_id",
    "names",
    "mechanic",
    "category",
    "designer",
    "mechanic_list",
    "category_list"
]

X = df.drop(columns=drop_cols + [target])
y = df[target]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

Feature matrix shape: (4984, 154)
Target vector shape: (4984,)


In [25]:
#The final step verifies that the processed dataset contains no missing values and confirms that the feature matrix is ready for training machine learning models.
print("Missing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())

display(X.head())
display(y.head())

Missing values in X: 0
Missing values in y: 0


,min_players,max_players,avg_time,min_time,max_time,year,geek_rating,num_votes,age,owned,...,Wargame,Word Game,World War I,World War II,Zombies,none,designer_avg_rating,designer_num_games,designer_total_votes,mechanic_popularity
0,1,4,120,60,120,2017,8.61858,15376,12,25928,...,0,0,0,0,0,0,8.190565,2,16648,482.222222
1,2,4,60,60,60,2015,8.50163,26063,13,41605,...,0,0,0,0,0,0,8.563245,2,30244,609.285714
2,2,4,240,180,240,2015,8.30183,12352,14,15848,...,0,0,0,0,0,0,7.171458,19,174603,467.333333
3,1,5,120,120,120,2016,8.19914,26004,12,33340,...,0,0,0,0,0,0,7.482233,3,26563,842.800000
4,2,2,180,120,180,2005,8.19787,31301,13,42952,...,1,0,0,0,0,0,8.339540,1,31301,804.000000


,avg_rating
0,8.98893
1,8.66140
2,8.60673
3,8.38461
4,8.33954


In [27]:
X.to_csv("data_cleaned_engineered_input.csv", index=False)
y.to_csv("data_cleaned_engineered_target.csv", index=False)